# Gold table


In [ ]:
from pathlib import Path

import pandas as pd

EXPERIMENT = "gulf_stream_pigment_influencers_20241001_20251231"
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
gold_table_path = repo_root / "data" / EXPERIMENT / "gold" / "eddy_pigment_table.parquet"

gold = pd.read_parquet(gold_table_path)
print(gold)

In [ ]:
gold.columns

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

target_cols = [x for x in list(gold.columns) if x.startswith("log_ratio")]
predictor_cols = ["polarity", "season", "movement", "age_frac"]
X = pd.get_dummies(gold[predictor_cols])
y = gold[target_cols]
# Remove rows w/ nans since that messes up Random Forest Regression
mask = y.notna().all(axis=1) # notna() returns same shape, .all(axis=1) collapses across columns returning a series of bools of length (rows)
X, y = X[mask], y[mask]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026)

# group one-hot columns back under their predictor (season_DJF -> season)
col_to_group = {c: next(p for p in predictor_cols if c == p or c.startswith(p + "_")) for c in X.columns}

models = {}
importances = {}
for col in target_cols:
    model = RandomForestRegressor(random_state=2026)
    model.fit(X_train, y_train[col])
    models[col] = model
    imp = pd.Series(model.feature_importances_, index=X.columns)
    importances[col] = imp.groupby(col_to_group).sum().sort_values(ascending=False)

for target, model in models.items():
    test_r2 = model.score(X_test, y_test[target])
    print(
        f"target: {target}\n"
        f"r2: {test_r2}"
    )

print("---------------------------------------------------------------")

for col in target_cols:
    print(f"{col} predictor importances:")
    print(importances[col])

# A stronger forest

The forest above reaches at most R^2 ~0.15, and goes negative for DV_chla.
Two limits apply.
The default `RandomForestRegressor` grows unbounded trees that memorize individual eddy-days.
The four predictors leave out each eddy's strength, size, and position.

Below is the same one-forest-per-pigment setup with the trees regularized and those features added.
Scoring also switches to GroupKFold by `track_id`.
Only 92 eddies make up the ~1290 eddy-days, so a random split would put the same eddy in train and test and inflate the result.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import r2_score

target_cols = [c for c in gold.columns if c.startswith("log_ratio")]
predictor_cols = [
    # class and lifecycle
    "polarity", "season", "movement", "age_frac",
    # strength, size, rotation
    "amplitude_cm", "radius_km", "age_days",
    "rossby_center", "rossby_abs_mean", "rossby_min", "rossby_max",
    # where it sits
    "gs_dist_km", "center_lat", "center_lon",
]
X = pd.get_dummies(gold[predictor_cols])
y = gold[target_cols].replace([np.inf, -np.inf], np.nan)
mask = y.notna().all(axis=1)  # drop the two eddy-days whose eddy mean is 0
X, y, groups = X[mask], y[mask], gold.loc[mask, "track_id"]

# 92 eddies make up these eddy-days, so a random split would leak the same eddy into train and test.
# GroupKFold keeps every eddy on one side of each split.
rf = RandomForestRegressor(random_state=2026, n_estimators=400,
    max_depth=8, min_samples_leaf=3, max_features=0.5)
gkf = GroupKFold(n_splits=5)
r2 = pd.Series({
    col: r2_score(y[col], cross_val_predict(rf, X, y[col], groups=groups, cv=gkf, n_jobs=-1))
    for col in target_cols
}).sort_values(ascending=False)
print(r2.round(3).to_string())
print(f"mean_r2: {r2.mean():.3f}")

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

# Impurity importance is free from the fit but inflates predictors with many split points (movement's four classes, the season one-hots).
# Permutation importance shuffles each predictor on held-out eddies and measures the real drop in R^2, so reading the two side by side is more honest than trusting either alone.
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=2026).split(X, y, groups))
col_to_group = {c: next(p for p in predictor_cols if c == p or c.startswith(p + "_")) for c in X.columns}

impurity_scores, perm_scores = [], []
for col in target_cols:
    fit = rf.fit(X.iloc[train_idx], y[col].iloc[train_idx])
    impurity_scores.append(pd.Series(fit.feature_importances_, index=X.columns).groupby(col_to_group).sum())
    pi = permutation_importance(fit, X.iloc[test_idx], y[col].iloc[test_idx],
        n_repeats=10, random_state=2026, n_jobs=-1)
    perm_scores.append(pd.Series(pi.importances_mean, index=X.columns).groupby(col_to_group).sum())

importances = pd.DataFrame({
    "impurity": pd.concat(impurity_scores, axis=1).mean(axis=1),
    "permutation": pd.concat(perm_scores, axis=1).mean(axis=1),
}).sort_values("permutation", ascending=False)
print(importances.round(3).to_string())

Mean out-of-fold R^2 is now about 0.48, even under the stricter grouped scoring.
Most of the jump is the regularization (shallower trees, `max_features=0.5`) that reduces the overfitting.
The added features, mainly the position ones, supply the rest.

Impurity importance is what the trees split on, and it favors predictors with many split points, so `movement` (four classes) looks like a top-three driver.
Permutation importance instead shuffles each predictor on held-out eddies and measures the actual drop in R^2.
By that measure distance to the Gulf Stream and season lead, latitude follows, and `movement`, `polarity`, eddy strength, and size all collapse toward zero.

So the eddy's own class barely helps.
Dropping both `polarity` and `movement` leaves held-out R^2 unchanged (~0.45), so the earlier "polarity matters" read was mostly impurity bias.
One nuance: permutation importance is marginal given the rest.
Strength and size reading near zero partly means position already carries their information, not that they are meaningless on their own.

The standing caveat: position encodes the regional pigment gradient across the Gulf Stream front, not necessarily the eddy's own effect.
The log-ratio already divides by the local background.

# Empirical pigment fractions on PACE L3 chlorophyll

The 7/14 update trained one forest per pigment on the SDP retrieval, that is the
`eddy_mean_{pigment}` columns of the gold table. This section changes one thing
and holds everything else fixed: the pigment field. Each pigment becomes a share
of total chlorophyll a,

    pigment = fraction x Tchla

The fraction is empirical. It comes from HPLC casts inside the region of
interest. Tchla is `chlor_a` from the PACE OCI Level-3 8-day BGC product, that
is the standard OCI band-ratio chlorophyll, read on the same pixels SDP ran on.
Same eddies, same dates, same masks, same predictors, same forest settings.

Two fraction models run side by side:

1. `const`: `pigment = f * chlor_a`, one number `f` per pigment. This is a skill
   floor, not a usable method, because measuring `f` needs the bottles you would
   be scoring against.
2. `power`: `pigment = a * chlor_a**b`, that is a straight line in log-log space,
   so the fraction itself moves with chlorophyll. The HPLC casts show it does.
   This is the real competitor. It needs an HPLC archive and an operational
   `chlor_a`, and no hyperspectral retrieval at all.

One difference from 7/14 is deliberate. 7/14 scored the R2 tables on a random
`train_test_split`. Every score below instead comes from `GroupKFold(track_id)`,
because one eddy supplies many eddy-days and a random split puts the same eddy
on both sides.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

repo_root = Path("/Users/jerry/school/research/eddy-tracking")
for extra in (repo_root, repo_root / "src", repo_root / "scripts"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from eddy_tracking.validation.seabass import read_hplc_dir

EXPERIMENT = "gulf_stream_pigment_influencers_20241001_20251231_v1"
ROSSBY_ABS_MEAN_THRESHOLD = 0.1
REGION_LON = (-81, -56)
REGION_LAT = (29, 44)
SURFACE_DEPTH_M = 10
# SDP and the OCI chlorophyll both invert a marine model, so the freshwater cruises are out of scope for a fraction measured against them.
FRESHWATER_EXPERIMENTS = ("PVST_PRINGLS", "PVST_GLOC")
# SeaBASS HPLC field -> gold-table pigment suffix.
SEABASS_TO_PIGMENT = {
    "zea": "Zea", "dv_chl_a": "DV_chla", "but-fuco": "ButFuco",
    "hex-fuco": "HexFuco", "allo": "Allo", "mv_chl_b": "MV_chlb",
    "neo": "Neo", "viola": "Viola", "fuco": "Fuco",
    "chl_c1c2": "Chlc12", "chl_c3": "Chlc3", "perid": "Perid",
}
PIGMENTS = ["Tchla"] + list(SEABASS_TO_PIGMENT.values())

# below_detection="zero" keeps a cast whose pigment sat under the detection limit.
# Dropping those casts would average only the casts where the pigment was found, which biases every fraction upward.
casts = pd.concat(
    [read_hplc_dir(repo_root / "data" / name, below_detection="zero")
        for name in ("pvst_hplc", "pvst_bats_hplc")],
    ignore_index=True,
)
casts = casts[
    (casts["depth"] <= SURFACE_DEPTH_M)
    & ~casts["experiment"].isin(FRESHWATER_EXPERIMENTS)
    & casts["lon"].between(*REGION_LON)
    & casts["lat"].between(*REGION_LAT)
    & (casts["tot_chl_a"] > 0)
]
print(f"surface_hplc_casts_in_region: {len(casts)}")
print(casts.groupby("experiment").size().to_string())
print(
    f"\ncast_tchla_mg_m3_median: {casts['tot_chl_a'].median():.3f}\n"
    f"cast_tchla_mg_m3_min: {casts['tot_chl_a'].min():.3f}\n"
    f"cast_tchla_mg_m3_max: {casts['tot_chl_a'].max():.3f}"
)

cast_tchla = casts["tot_chl_a"].to_numpy()
fraction_rows = {}
for field, pigment in SEABASS_TO_PIGMENT.items():
    measured = casts[field].to_numpy()
    detected = measured > 0
    # The power law needs logs, so it can only use casts above the detection limit.
    slope, intercept = np.polyfit(
        np.log(cast_tchla[detected]), np.log(measured[detected]), 1
    )
    fraction_rows[pigment] = {
        "n_casts": len(measured),
        "n_detected": int(detected.sum()),
        # Ratio of sums, not the mean of the per-cast ratios.
        # It conserves total pigment over the cast set, it does not blow up on casts sitting near the chlorophyll floor, and it weights the casts whose chlorophyll is closest to the eddy range.
        "fraction": float(measured.sum() / cast_tchla.sum()),
        "median_ratio": float(np.median(measured / cast_tchla)),
        "power_a": float(np.exp(intercept)),
        "power_b": float(slope),
    }
fractions = pd.DataFrame(fraction_rows).T
print()
print(fractions.round(4).to_string())

`fraction` is the empirical share of total chlorophyll a each pigment carries.
HexFuco and Zea lead, near 0.2 each, which is what a subtropical community with
haptophytes and cyanobacteria looks like.

`n_detected` is the warning label. Allo sits above the detection limit in about
half the casts and Neo in about two thirds, so their fractions rest on far less
evidence than HexFuco or Chlc12.

`power_b` is the exponent in `pigment = a * chlor_a**b`. A value under 1 means
the pigment's share falls as chlorophyll rises, and a value over 1 means it
climbs. Fuco and MV_chlb sit near 1.2, that is diatom and green-algal pigments
gaining share in richer water. Viola sits near 0.62, losing share. None of them
sit at exactly 1, so the constant-fraction model is an approximation the casts
do not support.

In [ ]:
from collocate_pace_chl import load_or_collocate

# One PACE L3 chlor_a mean per eddy-day, over the same interior pixels SDP used, plus the per-date background mean over the same calm-water rule background.py applies.
# Cached under silver/pace_chl; the first run takes a few minutes.
chlorophyll = load_or_collocate(EXPERIMENT, fractions["power_b"].to_dict())

gold = pd.read_parquet(
    repo_root / "data" / EXPERIMENT / "gold" / "eddy_pigment_table.parquet"
)
gold = gold.loc[gold["rossby_abs_mean"] >= ROSSBY_ABS_MEAN_THRESHOLD]
table = gold.merge(chlorophyll, on=["track_id", "date", "polarity"], how="inner")
table = table.reset_index(drop=True)

print(
    f"eddy_days_passing_rossby_threshold: {len(gold)}\n"
    f"rossby_abs_mean_threshold: {ROSSBY_ABS_MEAN_THRESHOLD}"
)
print(
    f"eddy_days_with_pace_l3_chlorophyll: {len(table)}\n"
    f"tracks: {table['track_id'].nunique()}"
)
print()
print("pixels behind each mean")
print(table[["n_eddy_sdp_pixels", "n_eddy_chl_pixels",
    "n_bg_pixels", "n_bg_chl_pixels"]].describe().round(0).to_string())

The eddy pixel counts agree to within a few pixels, so the two pigment fields
really are averaged over the same water. The background counts do not, on
purpose: `background.py` subsamples to 2000 pixels because SDP is slow per
pixel, while `chlor_a` is already computed, so every calm pixel enters its mean.

Before the forests, the two chlorophyll estimates need a look. They come from
the same instrument on the same day over the same pixels, so any disagreement is
algorithm, not sampling.

In [ ]:
import matplotlib.pyplot as plt

CHLOROPHYLL_TICKS = [0.03, 0.1, 0.3, 1.0]
# The background is one value per composite date, so plotting it per eddy-day would draw the same 58 points hundreds of times over.
by_date = table.drop_duplicates("date")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), constrained_layout=True)
for ax, (frame, sdp_col, chl_col, title) in zip(
    axes,
    [(table, "eddy_mean_Tchla", "eddy_chl_mean", f"Eddy interior (n={len(table):,})"),
        (by_date, "bg_mean_Tchla", "bg_chl_mean", f"Background (n={len(by_date)} dates)")],
):
    sdp = frame[sdp_col].to_numpy()
    chl = frame[chl_col].to_numpy()
    ax.scatter(sdp, chl, s=14, alpha=0.3, color="#2f6f9f", linewidths=0)
    limits = (0.02, 1.5)
    ax.plot(limits, limits, color="0.35", linewidth=1, linestyle="--", zorder=0)
    correlation = np.corrcoef(np.log(sdp), np.log(chl))[0, 1]
    ax.set_title(f"{title}\nlog-log r = {correlation:.2f}, "
        f"median chlor_a / Tchla = {np.median(chl / sdp):.2f}", fontsize=11)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(limits)
    ax.set_ylim(limits)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("SDP Tchla (mg m$^{-3}$)", fontsize=10)
    ax.set_xticks(CHLOROPHYLL_TICKS, [str(t) for t in CHLOROPHYLL_TICKS])
    ax.set_yticks(CHLOROPHYLL_TICKS, [str(t) for t in CHLOROPHYLL_TICKS])
    ax.minorticks_off()
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("PACE L3 chlor_a (mg m$^{-3}$)", fontsize=10)
fig.suptitle("Two chlorophyll estimates from the same PACE pixels", fontsize=12)
plt.show()

In [ ]:
# pigment = fraction * chlor_a, so the eddy mean is fraction * the eddy mean of chlor_a.
# The power model needs the mean of chlor_a**b over the pixels, which collocate_pace_chl already averaged pixel by pixel.
for pigment in SEABASS_TO_PIGMENT.values():
    fraction = fractions.loc[pigment, "fraction"]
    scale = fractions.loc[pigment, "power_a"]
    table[f"const_eddy_{pigment}"] = fraction * table["eddy_chl_mean"]
    table[f"const_bg_{pigment}"] = fraction * table["bg_chl_mean"]
    table[f"power_eddy_{pigment}"] = scale * table[f"eddy_chl_pow_{pigment}"]
    table[f"power_bg_{pigment}"] = scale * table[f"bg_chl_pow_{pigment}"]

# Tchla is the chlorophyll itself, so its fraction is 1 and its exponent is 1.
for family in ("const", "power"):
    table[f"{family}_eddy_Tchla"] = table["eddy_chl_mean"]
    table[f"{family}_bg_Tchla"] = table["bg_chl_mean"]

# The three target types 7/14 compared, for one pigment field.
PIGMENT_FIELDS = {
    "sdp": ("eddy_mean_", "bg_mean_"),
    "const": ("const_eddy_", "const_bg_"),
    "power": ("power_eddy_", "power_bg_"),
}


def build_targets(frame, field):
    """The mean, ratio, and log-ratio target of every pigment for one field."""
    eddy_prefix, bg_prefix = PIGMENT_FIELDS[field]
    eddy = {p: frame[f"{eddy_prefix}{p}"] for p in PIGMENTS}
    background = {p: frame[f"{bg_prefix}{p}"] for p in PIGMENTS}
    return {
        "mean": eddy,
        "ratio": {p: eddy[p] / background[p] for p in PIGMENTS},
        "log_ratio": {p: np.log(eddy[p] / background[p]) for p in PIGMENTS},
    }


targets = {field: build_targets(table, field) for field in PIGMENT_FIELDS}
# one bool row per pigment target: (n_targets, n_eddy_days) -> (n_eddy_days,)
finite = np.all(
    [np.isfinite(series) for field in targets.values()
        for kind in field.values() for series in kind.values()],
    axis=0,
)
print(
    f"fully_finite_eddy_days: {finite.sum()}\n"
    f"total_eddy_days: {len(table)}"
)
table = table[finite].reset_index(drop=True)
targets = {field: build_targets(table, field) for field in PIGMENT_FIELDS}
print(
    f"modeling_eddy_days: {len(table)}\n"
    f"tracks: {table['track_id'].nunique()}"
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, cross_val_predict

BINARY_PREDICTORS = ["polarity"] # already 0/1
CATEGORICAL_PREDICTORS = ["movement"] # expanded into one-hot columns
DESIGN_NUMERIC = ["age_frac", "time_of_year_cos", "time_of_year_sin"]
FULL_NUMERIC = DESIGN_NUMERIC + [
    # strength, size, rotation
    "amplitude_cm", "radius_km", "age_days",
    "rossby_center", "rossby_abs_mean", "rossby_min", "rossby_max",
    # where it sits
    "gs_dist_km", "center_lat", "center_lon",
]
PREDICTOR_SETS = {"design": DESIGN_NUMERIC, "full": FULL_NUMERIC}


def build_design(frame, numeric_cols):
    """One-hot design matrix, plus the map that folds importance back per predictor."""
    predictor_cols = BINARY_PREDICTORS + CATEGORICAL_PREDICTORS + numeric_cols
    X = pd.get_dummies(frame[predictor_cols], columns=CATEGORICAL_PREDICTORS)
    # One-hot columns are named like movement_SS; the others keep their name.
    col_to_group = {
        col: group for group in predictor_cols for col in X.columns
        if col == group or col.startswith(group + "_")
    }
    return X, predictor_cols, col_to_group


def build_forest():
    """The 7/14 regularized forest."""
    return RandomForestRegressor(
        random_state=2026, n_estimators=400, max_depth=8,
        min_samples_leaf=3, max_features=0.5,
    )


def score_targets(frame, target_by_name, numeric_cols):
    """Out-of-fold R2 and impurity importance per target, grouped by track_id."""
    X, predictor_cols, col_to_group = build_design(frame, numeric_cols)
    groups = frame["track_id"]
    splitter = GroupKFold(n_splits=5)
    rows, predicted = {}, {}
    for name, y in target_by_name.items():
        out_of_fold = cross_val_predict(
            build_forest(), X, y, groups=groups, cv=splitter, n_jobs=-1
        )
        fitted = build_forest().fit(X, y)
        importance = (
            pd.Series(fitted.feature_importances_, index=X.columns)
            .groupby(col_to_group).sum()
        )
        rows[name] = {"r2": r2_score(y, out_of_fold),
            **{f"imp_{p}": importance[p] for p in predictor_cols}}
        predicted[name] = out_of_fold
    return pd.DataFrame(rows).T, predicted


scores, out_of_fold = {}, {}
for predictor_set, numeric_cols in PREDICTOR_SETS.items():
    for field, target_types in targets.items():
        for target_type, target_by_name in target_types.items():
            key = (predictor_set, field, target_type)
            scores[key], out_of_fold[key] = score_targets(
                table, target_by_name, numeric_cols
            )
            print(
                f"predictor_set: {predictor_set}\n"
                f"field: {field}\n"
                f"target_type: {target_type}\n"
                f"mean_r2: {scores[key]['r2'].mean():+.3f}",
                flush=True,
            )

In [ ]:
r2_summary = pd.DataFrame({
    f"{predictor_set}/{target_type}": {
        field: scores[(predictor_set, field, target_type)]["r2"].mean()
        for field in PIGMENT_FIELDS
    }
    for predictor_set in PREDICTOR_SETS
    for target_type in ("mean", "ratio", "log_ratio")
})
print("mean out-of-fold R2 over the 13 pigments")
print(r2_summary.round(3).to_string())

print("\n\nper-pigment R2, full predictors, mean target")
print(pd.DataFrame({
    field: scores[("full", field, "mean")]["r2"] for field in PIGMENT_FIELDS
}).round(3).to_string())

print("\n\nper-pigment R2, full predictors, log-ratio target")
print(pd.DataFrame({
    field: scores[("full", field, "log_ratio")]["r2"] for field in PIGMENT_FIELDS
}).round(3).to_string())

The `const` column repeats one number down all 13 pigments. That is not a bug,
it is the whole content of a constant fraction.

A random forest splits on squared-error reduction. Multiply a target by a
positive constant and every candidate split keeps its rank, so the tree
structure, the fold assignment, and the leaf means all scale by that same
constant. R2 is scale free, so in exact arithmetic it does not move, and all
that survives is rounding inside the split search. The spread table below
measures how little that is: the 13 `const` pigments span 0.002 of R2 on the
mean target and 0.0003 on the log ratio, against 0.32 and 0.45 for SDP.

For the ratio and log-ratio targets the point is sharper still. The fraction
divides out, `f * eddy / (f * background) = eddy / background`, so all 13
targets are one column of numbers.

`power` breaks the tie only on the mean target, where `mean(chlor_a**b)` over
the eddy pixels really is a different eddy-day statistic for each `b`. That
opens the spread to 0.08. On the log ratio the exponent nearly factors out,
because `log(a * mean(T_eddy**b)) - log(a * mean(T_bg**b))` sits close to `b`
times the chlorophyll log ratio, and a constant multiple takes R2 nowhere. The
spread stays at 0.003.

So no fraction-of-chlorophyll model can answer "which pigment responds to
eddies". Whatever fraction goes in, the question it answers is "how predictable
is chlorophyll", asked 13 times.

In [ ]:
# Check the scaling argument instead of trusting it: how far do the 13 pigments spread inside each field?
spread = pd.DataFrame({
    f"{field}/{target_type}": {
        "min_r2": scores[("full", field, target_type)]["r2"].min(),
        "max_r2": scores[("full", field, target_type)]["r2"].max(),
        "spread": np.ptp(scores[("full", field, target_type)]["r2"]),
    }
    for field in PIGMENT_FIELDS
    for target_type in ("mean", "log_ratio")
})
print("R2 spread across the 13 pigments, full predictors")
print(spread.round(4).to_string())

importance_summary = pd.DataFrame({
    field: scores[("full", field, "mean")].filter(like="imp_").mean()
    for field in PIGMENT_FIELDS
}).sort_values("sdp", ascending=False)
importance_summary.index = importance_summary.index.str.removeprefix("imp_")
print("\n\nmean impurity importance over the 13 pigments, full predictors, mean target")
print(importance_summary.round(3).to_string())

In [ ]:
from matplotlib.ticker import MaxNLocator

HEADLINE = ["Tchla", "Zea", "DV_chla", "ButFuco", "HexFuco"]

fig, axes = plt.subplots(2, len(HEADLINE), figsize=(19, 8.0), constrained_layout=True)
for row, field in enumerate(["sdp", "const"]):
    for ax, pigment in zip(axes[row], HEADLINE):
        observed = targets[field]["mean"][pigment].to_numpy()
        predicted = out_of_fold[("full", field, "mean")][pigment]
        ax.scatter(observed, predicted, s=14, alpha=0.35,
            color="#2f6f9f", linewidths=0)
        low = min(observed.min(), predicted.min())
        high = max(observed.max(), predicted.max())
        pad = 0.05 * (high - low)
        low, high = low - pad, high + pad
        ax.plot([low, high], [low, high], color="0.35", linewidth=1,
            linestyle="--", zorder=0)
        ax.set_title(f"{pigment}\nR$^2$={r2_score(observed, predicted):.2f}", fontsize=11)
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)
        ax.set_aspect("equal", adjustable="box")
        ax.tick_params(labelsize=9)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.spines[["top", "right"]].set_visible(False)
    axes[row][0].set_ylabel(
        f"{'SDP' if field == 'sdp' else 'Empirical fraction'}\n"
        "random forest (mg m$^{-3}$)", fontsize=10)

fig.supxlabel("Pigment field value (mg m$^{-3}$)", fontsize=11)
fig.suptitle(
    "Random forest against the pigment it was trained on\n"
    f"full predictors, track-grouped 5-fold cross-validation (n={len(table):,})",
    fontsize=12,
)
plt.show()

In [ ]:
# A forest score says how smooth a field is, not whether two fields agree.
# Compare them directly, splitting the comparison into level and pattern: the median ratio is the systematic offset, and the log-space r2 and multiplicative scatter describe what is left once that offset is taken out.
agreement = {}
for pigment in PIGMENTS:
    sdp = table[f"eddy_mean_{pigment}"].to_numpy()
    empirical = table[f"power_eddy_{pigment}"].to_numpy()
    log_residual = np.log(empirical) - np.log(sdp)
    agreement[pigment] = {
        "sdp_median": np.median(sdp),
        "emp_median": np.median(empirical),
        "median_ratio": np.median(empirical / sdp),
        "log_r2": np.corrcoef(np.log(sdp), np.log(empirical))[0, 1] ** 2,
        # A factor, so 1.5 means the two fields sit within x1.5 of each other about two thirds of the time once the median offset is removed.
        "scatter_factor": np.exp(np.std(log_residual)),
    }
agreement = pd.DataFrame(agreement).T
print("chlorophyll-law pigment against the SDP pigment, eddy interior means")
print(agreement.round(3).to_string())

sdp_shares = pd.Series({
    pigment: (table[f"eddy_mean_{pigment}"] / table["eddy_mean_Tchla"]).median()
    for pigment in SEABASS_TO_PIGMENT.values()
})
comparison = pd.DataFrame({
    "hplc_fraction": fractions["fraction"].astype(float),
    "sdp_median_share": sdp_shares,
})
comparison["sdp_over_hplc"] = comparison["sdp_median_share"] / comparison["hplc_fraction"]
print("\n\npigment share of Tchla: HPLC casts against the SDP field")
print(comparison.round(3).to_string())

In [ ]:
# How much of each SDP pigment is chlorophyll wearing a different name?
# A pigment held at a fixed share of chlorophyll has a log that tracks the log of chlorophyll exactly, so this r2 is the share a chlorophyll law already covers.
tracking = pd.DataFrame({
    pigment: {
        "vs_sdp_Tchla": np.corrcoef(
            np.log(table[f"eddy_mean_{pigment}"]), np.log(table["eddy_mean_Tchla"])
        )[0, 1] ** 2,
        "vs_pace_chlor_a": np.corrcoef(
            np.log(table[f"eddy_mean_{pigment}"]), np.log(table["eddy_chl_mean"])
        )[0, 1] ** 2,
    }
    for pigment in PIGMENTS
}).T
print("log-space r^2 of each SDP pigment against a chlorophyll field")
print(tracking.round(3).to_string())
print(f"\na chlorophyll law already covers: "
    f"{', '.join(tracking.index[tracking['vs_sdp_Tchla'] >= 0.7])}")
print(f"carries information beyond chlorophyll: "
    f"{', '.join(tracking.index[tracking['vs_sdp_Tchla'] < 0.2])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0), constrained_layout=True)

order = comparison.sort_values("hplc_fraction", ascending=False).index
positions = np.arange(len(order))
axes[0].barh(positions - 0.2, comparison.loc[order, "hplc_fraction"], height=0.38,
    color="#2f6f9f", label="HPLC casts")
axes[0].barh(positions + 0.2, comparison.loc[order, "sdp_median_share"], height=0.38,
    color="#c9772f", label="SDP field")
axes[0].set_yticks(positions)
axes[0].set_yticklabels(order, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel("Share of total chlorophyll a", fontsize=10)
axes[0].set_title("Where the two fields put the pigment", fontsize=11)
axes[0].legend(frameon=False, fontsize=9)
axes[0].spines[["top", "right"]].set_visible(False)
axes[0].tick_params(labelsize=9)

# Binned medians rather than raw points: three overlapping clouds of 901 points hide the trend that is the whole point of the panel.
chlorophyll_bins = pd.qcut(table["eddy_mean_Tchla"], 8)
highest_share = 0.0
for pigment, color in zip(["Zea", "HexFuco", "Fuco", "DV_chla"],
        ["#2f6f9f", "#c9772f", "#4c9a6b", "#8c6bb1"]):
    share = table[f"eddy_mean_{pigment}"] / table["eddy_mean_Tchla"]
    binned = share.groupby(chlorophyll_bins, observed=True).median()
    centres = table["eddy_mean_Tchla"].groupby(chlorophyll_bins, observed=True).median()
    highest_share = max(highest_share, binned.max())
    axes[1].plot(centres, binned, "o-", color=color, linewidth=1.8, markersize=5,
        label=f"{pigment}, SDP")
    axes[1].axhline(fractions.loc[pigment, "fraction"], color=color,
        linewidth=1.2, linestyle="--")
axes[1].set_xscale("log")
axes[1].set_xlim(0.05, 1.0)
axes[1].set_xticks([0.05, 0.1, 0.3, 1.0], ["0.05", "0.1", "0.3", "1.0"])
axes[1].minorticks_off()
axes[1].set_ylim(0, np.ceil(highest_share * 10) / 10)
axes[1].set_xlabel("SDP Tchla (mg m$^{-3}$), binned", fontsize=10)
axes[1].set_ylabel("Pigment / Tchla, median in bin", fontsize=10)
axes[1].set_title("Dashed line is the constant HPLC fraction", fontsize=11)
axes[1].legend(frameon=False, fontsize=9, loc="upper right")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].tick_params(labelsize=9)

fig.suptitle("Pigment share of chlorophyll: HPLC fraction against the SDP retrieval",
    fontsize=12)
plt.show()

## What the run says

**The chlorophyll baseline is not the weaker model. It wins, and by a lot.**
With the full predictor set the chlorophyll-based targets reach 0.79 out-of-fold
R2 on the mean and 0.85 on the log ratio, against 0.63 and 0.42 for the SDP
targets. The design-predictor run splits the same way, 0.65 against 0.50. A
forest that predicts `f * chlor_a` twice as well as it predicts SDP's pigment is
saying the SDP field holds a large share of variance that eddy position, season,
size, and strength do not reach. Part of that is pigment structure the baseline
throws away. Part of it is retrieval noise. This run does not separate the two,
but it does size the gap, and it means "SDP scored 0.42 on the log ratio" cannot
be read as "eddies explain 42% of the pigment ratio" without first asking why a
chlorophyll law reaches 0.85 on the same rows.

The background is not the cause. The chlorophyll background averages about
78,000 calm pixels per date while `background.py` subsamples 2,000 for SDP, so
the SDP log ratio does divide by a noisier number. That cannot be the main
effect, because the `mean` target never touches the background and SDP loses
there too, 0.63 against 0.79.

**A constant fraction carries no pigment-specific information at all.** The 13
`const` R2 values are one number repeated to within rounding, and on the ratio
targets the 13 targets are one column of numbers. Letting the fraction follow
chlorophyll as `a * chlor_a**b` separates them by 0.08 on the mean target and
0.003 on the log ratio. So the baseline cannot rank pigments, by construction.
It is a floor: any per-pigment claim from SDP has to clear a model that knows
only chlorophyll.

**Most SDP pigments are close to a rescaled chlorophyll. Three are not.** The
tracking table gives the log-space r2 of each SDP pigment against SDP Tchla:
Chlc3 0.98, Chlc12 0.95, Neo 0.89, HexFuco 0.83, Allo 0.81, and so on down. Zea
at 0.05, DV_chla at 0.03, and Perid at 0.17 are the exceptions. For the pigments
near the top of that list, an eddy result is a chlorophyll result with a
different label, and the honest per-pigment claims have to come from the bottom
three.

**Those same three are where SDP looks most predictable, and that is a warning
rather than a win.** SDP's best mean-target scores are Perid 0.83 and Zea 0.76,
well above the 0.51 to 0.67 the rest sit at, and its best log-ratio scores are
Perid 0.76 and Zea 0.64 against 0.31 to 0.50. A pigment that ignores chlorophyll
but tracks season and distance to the jet this closely is either a real
community signal or a smooth retrieval artifact. The station-level work already
found SDP Zea running backwards against HPLC, so the artifact reading needs
ruling out before either number is used.

**The two chlorophyll estimates disagree.** PACE L3 `chlor_a` and SDP Tchla come
from the same instrument, the same 8-day window, and the same pixels. Their
log-log correlation is about 0.48 in eddy interiors, and `chlor_a` runs near half
of SDP Tchla. Part of that is definition, since SDP's Tchla is monovinyl plus
divinyl chlorophyll a while OCI is a band-ratio fit, but a factor of two with
that much scatter is wider than a definition gap.

**Where the pigment sits differs too.** `sdp_over_hplc` compares SDP's median
share of Tchla against the cast-derived fraction. SDP puts about twice the Fuco
the casts do, 1.7 times the Allo, and 1.6 times the Neo, while it puts less Zea,
at 0.71. The binned panel shows why one fraction cannot stand in for the field
either way: SDP's share slides with chlorophyll instead of holding flat.

**What this does not settle.** These are bulk eddy-day means. Nothing here
speaks to radial structure inside an eddy, which is where polarity did show an
effect, and the same question applies there: check the radial gradient against a
chlorophyll law before reading it as a pigment result.